# exp1_pix2pix vs exp2_paper

Visual read-out of the two finished 200-epoch MRI→CT runs, from the artifacts in
`exp1_pix2pix_results/` and `exp2_paper_results/`.

| section | what it shows |
|---|---|
| Headline metrics | MAE (HU), PSNR, SSIM, bone Dice over training |
| Per-region | the same metrics split by `brain / spine / abdomen / musculoskeletal` |
| HU bands | soft-tissue vs bone-window error |
| Training + GAN health | `G_L1`, `G_GAN`, discriminator loss and accuracy |
| NCE | exp2's per-layer PatchNCE loss |
| Scorecard | every metric at each run's own best epoch |
| Panels | the MRI / real CT / synth CT / \|error\| sample grids |

## Setup

In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

# Resolve the repo root by walking up, so the notebook runs from either
# model/notebooks/ or the repo root without editing paths.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "exp1_pix2pix_results").is_dir())

RUNS = {
    "exp1_pix2pix": ROOT / "exp1_pix2pix_results",
    "exp2_paper":   ROOT / "exp2_paper_results",
}
SHORT = {"exp1_pix2pix": "exp1", "exp2_paper": "exp2"}

# One fixed hue per run, reused by every chart below. Colour follows the run and
# never its rank in a sort, so no chart repaints when the ordering changes.
COLORS = {"exp1_pix2pix": "#2a78d6", "exp2_paper": "#eb6834"}

INK, INK_MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"

mpl.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": GRID, "axes.linewidth": 1.0,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.labelcolor": INK_MUTED, "text.color": INK,
    "xtick.color": INK_MUTED, "ytick.color": INK_MUTED,
    "xtick.labelsize": 9, "ytick.labelsize": 9,
    "axes.titlesize": 10.5, "axes.titlelocation": "left",
    "legend.frameon": False, "lines.linewidth": 2.0,
    "figure.dpi": 110, "savefig.bbox": "tight",
})

print("repo root:", ROOT)
for _name, _d in RUNS.items():
    print(f"  {_name:14s} {'found' if _d.is_dir() else 'MISSING'}  ->  {_d}")

In [ ]:
# Read train_log.jsonl, NOT metrics.csv.
#
# metrics.csv is rebuilt from this JSONL each epoch, but exp1 was trained before
# the fix documented in model/training/trainer.py::_log_epoch: its CSV header was
# frozen at epoch 0, which is GAN warm-up, so every train/D_* and train/G_GAN
# column is absent from it (54 columns, against exp2's 68). The JSONL is the
# authoritative record and has those keys for both runs.
logs = {name: pd.read_json(d / "train_log.jsonl", lines=True)
        for name, d in RUNS.items()}


def best_epoch(df):
    """Last epoch flagged is_best. Selection metric is val/mae_norm, minimised."""
    flagged = df.index[df["is_best"].fillna(False).astype(bool)]
    return int(df.loc[flagged[-1], "epoch"]) if len(flagged) else int(df["epoch"].iloc[-1])


BEST = {name: best_epoch(df) for name, df in logs.items()}

pd.DataFrame({
    name: {
        "epochs": len(df),
        "best epoch": BEST[name],
        "train time (min)": round(df["train_time_s"].sum() / 60, 1),
        "s / epoch": round(df["train_time_s"].mean(), 1),
        "scalars logged": df.shape[1],
    }
    for name, df in logs.items()
}).T.convert_dtypes()   # keep epoch counts as ints rather than 200.0

In [ ]:
def nudge_apart(ax, points, min_gap=0.062):
    """Push end labels apart vertically where lines converge. points = [[text, x, y]]."""
    if len(points) < 2:
        return points
    lo, hi = ax.get_ylim()
    span = (hi - lo) or 1.0
    points = sorted(points, key=lambda p: p[2])
    for lower, upper in zip(points, points[1:]):
        deficit = min_gap - (upper[2] - lower[2]) / span
        if deficit > 0:
            lower[2] -= deficit * span / 2
            upper[2] += deficit * span / 2
    return points


def end_labels(ax, col, pad_frac=0.18):
    """Direct-label each line at its right end, so identity is never colour-alone."""
    xmax = max(float(df["epoch"].iloc[-1]) for df in logs.values())
    ax.set_xlim(left=ax.get_xlim()[0], right=xmax * (1 + pad_frac))
    points = []
    for name, df in logs.items():
        if col not in df.columns:
            continue
        s = df[col].dropna()
        if s.empty:
            continue
        points.append([SHORT[name], float(df.loc[s.index[-1], "epoch"]), float(s.iloc[-1])])
    for text, x, y in nudge_apart(ax, points):
        ax.annotate(text, xy=(x, y), xytext=(7, 0), textcoords="offset points",
                    va="center", fontsize=8.5, color=INK_MUTED)


def series(ax, col, hint=None):
    """Draw both runs for one column, plus a marker on each run's best epoch."""
    for name, df in logs.items():
        if col not in df.columns:
            continue
        ax.plot(df["epoch"], df[col], color=COLORS[name], label=SHORT[name])
        row = df.loc[df["epoch"] == BEST[name], col]
        if not row.empty and pd.notna(row.iloc[0]):
            # The epoch that would actually be shipped, not the last one.
            ax.plot(BEST[name], row.iloc[0], "o", ms=7, color=COLORS[name],
                    mec="white", mew=2, zorder=5)
    end_labels(ax, col)
    if hint:
        ax.text(0, 1.02, hint, transform=ax.transAxes, fontsize=8, color=INK_MUTED)


def run_legend(fig, note="filled dot = best epoch (val/mae_norm)"):
    handles = [mpl.lines.Line2D([], [], color=COLORS[n], lw=2, label=SHORT[n])
               for n in logs]
    fig.legend(handles=handles, loc="lower center", ncol=len(handles) + 1,
               bbox_to_anchor=(0.5, -0.02), fontsize=9)
    fig.text(0.5, -0.055, note, ha="center", fontsize=8, color=INK_MUTED)

## Headline validation metrics

All four are computed on the same 230-slice validation set every epoch.

In [ ]:
HEADLINE = [
    ("val/mae_hu",    "Validation MAE (HU)",  "lower is better"),
    ("val/psnr",      "Validation PSNR (dB)", "higher is better"),
    ("val/ssim",      "Validation SSIM",      "higher is better"),
    ("val/dice_bone", "Bone Dice (>150 HU)",  "higher is better"),
]

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, (col, title, hint) in zip(axes.ravel(), HEADLINE):
    series(ax, col, hint)
    ax.set_title(title, pad=20)
for ax in axes[1]:
    ax.set_xlabel("epoch")
fig.tight_layout(h_pad=2.4, w_pad=3)
run_legend(fig)
plt.show()

## Per-region breakdown

`brain` is excluded from the bone-derived metrics by `eval.bone_metrics_exclude_regions`,
so those charts show three regions rather than four.

In [ ]:
REGIONS = ["brain", "spine", "abdomen", "musculoskeletal"]


def plot_by_region(base, title, hint=None):
    present = [r for r in REGIONS
               if any(f"val/{base}/{r}" in df.columns for df in logs.values())]
    fig, axes = plt.subplots(1, len(present), figsize=(3.1 * len(present), 3.2),
                             sharey=True)
    for ax, region in zip(np.atleast_1d(axes), present):
        series(ax, f"val/{base}/{region}")
        ax.set_title(region, pad=6)
        ax.set_xlabel("epoch")
    fig.suptitle(f"{title}" + (f"  —  {hint}" if hint else ""),
                 x=0.005, ha="left", fontsize=11)
    fig.tight_layout(w_pad=1.6)
    run_legend(fig)
    plt.show()


plot_by_region("mae_hu", "MAE by body region (HU)", "lower is better")
plot_by_region("ssim", "SSIM by body region", "higher is better")
plot_by_region("dice_bone", "Bone Dice by body region", "higher is better")

## HU-band error

Error restricted to the soft-tissue window (-200 to 150 HU) and the bone window
(150 HU and up), from `eval.hu_bands`. Separate axes — the two bands differ by
roughly an order of magnitude and do not belong on one scale.

In [ ]:
BANDS = [
    ("val/mae_band_soft", "MAE, soft-tissue band -200..150 HU"),
    ("val/mae_band_bone", "MAE, bone band >150 HU"),
]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for ax, (col, title) in zip(axes, BANDS):
    series(ax, col, "lower is better")
    ax.set_title(title, pad=20)
    ax.set_xlabel("epoch")
fig.tight_layout(w_pad=3)
run_legend(fig)
plt.show()

## Training losses and GAN health

`train.gan_warmup_epochs` is 5 in both runs, so the adversarial terms are absent
for epochs 0–4. The gaps below are that warm-up, left unfilled rather than
interpolated.

In [ ]:
LOSSES = [
    ("train/G_L1",    "Generator L1 (train)"),
    ("train/G_GAN",   "Generator adversarial loss (train)"),
    ("train/G_total", "Generator total loss (train)"),
    ("train/D_total", "Discriminator loss (train)"),
]

fig, axes = plt.subplots(2, 2, figsize=(11, 6.4))
for ax, (col, title) in zip(axes.ravel(), LOSSES):
    series(ax, col)
    ax.set_title(title, pad=6)
for ax in axes[1]:
    ax.set_xlabel("epoch")
fig.tight_layout(h_pad=2.0, w_pad=3)
run_legend(fig)
plt.show()

In [ ]:
# Discriminator accuracy, one panel per run: real and fake share the run's hue and
# are separated by line style, so the run stays identifiable across both panels.
# The dotted 0.5 line is the point where D can no longer tell them apart.
fig, axes = plt.subplots(1, len(logs), figsize=(11, 3.4), sharey=True)
for ax, (name, df) in zip(np.atleast_1d(axes), logs.items()):
    points = []
    for col, style, tag in [("train/D_acc_real", "-", "real"),
                            ("train/D_acc_fake", "--", "fake")]:
        if col not in df.columns:
            continue
        s = df[col].dropna()
        ax.plot(df.loc[s.index, "epoch"], s, style, color=COLORS[name])
        points.append([tag, float(df.loc[s.index[-1], "epoch"]), float(s.iloc[-1])])
    ax.axhline(0.5, color=INK_MUTED, lw=1, ls=":", zorder=0)
    ax.set_xlim(right=float(df["epoch"].iloc[-1]) * 1.18)
    ax.set_ylim(-0.05, 1.05)
    ax.set_yticks([0.0, 0.25, 0.5, 0.75, 1.0])
    for text, x, y in nudge_apart(ax, points):
        ax.annotate(text, xy=(x, y), xytext=(7, 0), textcoords="offset points",
                    va="center", fontsize=8.5, color=INK_MUTED)
    ax.set_title(f"{SHORT[name]} — discriminator accuracy", pad=6)
    ax.set_xlabel("epoch")
fig.legend(handles=[mpl.lines.Line2D([], [], color=INK_MUTED, lw=2, ls=s, label=t)
                    for s, t in [("-", "on real CT"), ("--", "on synth CT")]],
           loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.05), fontsize=9)
fig.tight_layout(w_pad=2.4)
plt.show()

## PatchNCE loss (exp2 only)

exp1 runs with `lambda_nce: 0.0`, so it logs no NCE terms and is skipped here.
Layer depth is an ordered quantity, so the five encoder layers use one hue
light→dark rather than five categorical colours.

In [ ]:
NCE_LAYERS = [f"train/G_NCE_L{i}" for i in range(5)]
ramp = plt.get_cmap("Blues")(np.linspace(0.38, 0.95, len(NCE_LAYERS)))

for name, df in logs.items():
    if "train/G_NCE" not in df.columns:
        print(f"{SHORT[name]}: no NCE terms logged (lambda_nce = 0) - skipped")
        continue

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))

    axes[0].plot(df["epoch"], df["train/G_NCE"], color=COLORS[name])
    axes[0].set_title(f"{SHORT[name]} — total PatchNCE loss", pad=6)

    # Every layer is directly labelled at its right end, so no legend box is needed.
    points = []
    for col, shade in zip(NCE_LAYERS, ramp):
        if col not in df.columns:
            continue
        axes[1].plot(df["epoch"], df[col], color=shade, lw=1.8)
        points.append([col.rsplit("_", 1)[-1], float(df["epoch"].iloc[-1]),
                       float(df[col].iloc[-1])])
    axes[1].set_xlim(right=float(df["epoch"].iloc[-1]) * 1.14)
    for text, x, y in nudge_apart(axes[1], points, min_gap=0.055):
        axes[1].annotate(text, xy=(x, y), xytext=(7, 0), textcoords="offset points",
                         va="center", fontsize=8.5, color=INK_MUTED)
    axes[1].set_title(f"{SHORT[name]} — PatchNCE by encoder layer (L0 shallow → L4 deep)",
                      pad=6)

    for ax in axes:
        ax.set_xlabel("epoch")
    fig.tight_layout(w_pad=3)
    plt.show()

## Scorecard at each run's best epoch

Each run is read at its **own** selected epoch, not at epoch 199 — that is the
checkpoint the training loop would hand you.

In [ ]:
SCORE = [
    ("val/mae_norm",       "MAE (normalised)",     "min"),
    ("val/mae_hu",         "MAE (HU)",             "min"),
    ("val/mae_hu__macro",  "MAE (HU, macro)",      "min"),
    ("val/psnr",           "PSNR (dB)",            "max"),
    ("val/ssim",           "SSIM",                 "max"),
    ("val/dice_bone",      "Bone Dice",            "max"),
    ("val/mae_band_soft",  "MAE soft band (HU)",   "min"),
    ("val/mae_band_bone",  "MAE bone band (HU)",   "min"),
]

rows = []
for col, label, direction in SCORE:
    vals = {}
    for name, df in logs.items():
        row = df.loc[df["epoch"] == BEST[name], col]
        vals[SHORT[name]] = float(row.iloc[0]) if not row.empty else np.nan
    delta = vals["exp2"] - vals["exp1"]
    rows.append({
        "metric": label, **vals, "delta (exp2-exp1)": delta,
        "better": ("exp2" if (delta < 0) == (direction == "min") else "exp1")
                  if np.isfinite(delta) else "",
    })

score = pd.DataFrame(rows).set_index("metric")


def _bold_winner(row):
    out = ["" for _ in row.index]
    if row["better"] in row.index:
        out[row.index.get_loc(row["better"])] = "font-weight:700"
    return out


score.style.apply(_bold_winner, axis=1).format(precision=4)

## Sample panels

Each panel is one row per fixed validation slice and four columns — MRI, real CT,
synth CT, and `|error|` on a fixed 0–0.5 scale. The same slices are rendered every
epoch, so panels compare directly across epochs and across runs.

In [ ]:
def sample_epochs(name):
    return sorted(int(p.stem.split("_")[1])
                  for p in (RUNS[name] / "samples").glob("epoch_*.png"))


def show_panel(name, epoch, width=9.0):
    img = Image.open(RUNS[name] / "samples" / f"epoch_{epoch:04d}.png")
    fig, ax = plt.subplots(figsize=(width, width * img.height / img.width))
    ax.imshow(img)
    ax.axis("off")
    ax.grid(False)
    ax.set_title(f"{name} — epoch {epoch}", pad=6)
    plt.show()


for _name in RUNS:
    show_panel(_name, sample_epochs(_name)[-1])

In [ ]:
# Progression at a glance: the same fixed slices at a spread of epochs.
def filmstrip(name, epochs=None, width=13.0):
    available = sample_epochs(name)
    epochs = epochs or [available[i] for i in
                        np.linspace(0, len(available) - 1, 5).astype(int)]
    fig, axes = plt.subplots(1, len(epochs), figsize=(width, width * 2.05 / len(epochs)))
    for ax, ep in zip(axes, epochs):
        ax.imshow(Image.open(RUNS[name] / "samples" / f"epoch_{ep:04d}.png"))
        ax.axis("off")
        ax.grid(False)
        ax.set_title(f"epoch {ep}", fontsize=9, loc="center")
    fig.suptitle(name, x=0.005, ha="left", fontsize=11)
    fig.tight_layout(w_pad=0.6)
    plt.show()


for _name in RUNS:
    filmstrip(_name)

### Epoch browser

Scrub through every saved panel for either run.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    EPOCHS = {name: sample_epochs(name) for name in RUNS}

    run_dd = widgets.Dropdown(options=list(RUNS), description="run:")
    ep_sl = widgets.SelectionSlider(
        options=EPOCHS[run_dd.value], value=EPOCHS[run_dd.value][-1],
        description="epoch:", continuous_update=False,
        layout=widgets.Layout(width="560px"))
    out = widgets.Output()

    def _render(*_):
        with out:
            out.clear_output(wait=True)
            show_panel(run_dd.value, ep_sl.value)

    def _on_run(change):
        ep_sl.options = EPOCHS[change["new"]]
        ep_sl.value = EPOCHS[change["new"]][-1]

    run_dd.observe(_on_run, names="value")
    ep_sl.observe(_render, names="value")

    display(widgets.VBox([widgets.HBox([run_dd, ep_sl]), out]))
    _render()
except ImportError:
    print("ipywidgets not installed - use the filmstrip cell above instead.")